In [51]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

fatal: destination path 'kcw-analytics' already exists and is not an empty directory.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using folder: /content/drive/Shareddrives


In [59]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [60]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "ITEMNO": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")



Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50266, 49)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154048, 41)
Loaded: raw_syp_pimas_purchase_bills.csv -> (2975, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (27820, 41)
Loaded: raw_syp_simas_sales_bills.csv -> (12944, 49)
Loaded: raw_syp_sidet_sales_lines.csv -> (37971, 38)
Loaded: raw_hq_icmas_products.csv -> (114956, 94)
Loaded: raw_hq_sidet_sales_lines.csv -> (733392, 38)
Loaded: raw_hq_apmas_payable.csv -> (978, 20)
Loaded: raw_hq_simas_sales_bills.csv -> (276005, 49)
Loaded: raw_hq_armas_receivable.csv -> (2227, 20)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13875, 32)


In [52]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw/statement"

In [53]:
!pip install xlrd

In [54]:
import os
import pandas as pd

data_statement = {}

for file in os.listdir(folder):
    path = os.path.join(folder, file)
    ext = os.path.splitext(file)[1].lower()

    try:
        if ext == ".csv":
            # text file
            last_err = None
            for enc in ["utf-8-sig", "cp874", "cp1252", "latin1"]:
                try:
                    df = pd.read_csv(path, skiprows=10, encoding=enc, low_memory=False)
                    source = f"csv-{enc}"
                    break
                except Exception as e:
                    last_err = e
            else:
                raise last_err

        elif ext == ".xlsx":
            # modern Excel
            df = pd.read_excel(path, header=10, engine="openpyxl")
            source = "xlsx-openpyxl"

        elif ext == ".xls":
            # old Excel binary format
            df = pd.read_excel(path, header=10, engine="xlrd")
            source = "xls-xlrd"

        else:
            continue

        for col in ["BCODE", "ITEMNO", "BILLNO"]:
            if col in df.columns:
                df[col] = df[col].astype("string")

        key = os.path.splitext(file)[0]
        data_statement[key] = df
        print(f"Loaded: {file} -> {df.shape} ({source})")

    except Exception as e:
        print(f"Failed: {file} -> {type(e).__name__}: {e}")

Loaded: raw_statement_6184.xls -> (31, 9) (xls-xlrd)
Loaded: raw_statement_0393.csv -> (67, 13) (csv-utf-8-sig)
Loaded: raw_statement_3557.csv -> (138, 13) (csv-utf-8-sig)
Loaded: raw_statement_7236.csv -> (92, 13) (csv-utf-8-sig)
Loaded: raw_statement_1139.xls -> (42, 9) (xls-xlrd)


In [55]:
df_6184 = data_statement["raw_statement_6184"].copy()
df_1139 = data_statement["raw_statement_1139"].copy()

df_0393 = data_statement["raw_statement_0393"].copy()
df_3557 = data_statement["raw_statement_3557"].copy()
df_7236 = data_statement["raw_statement_7236"].copy()


In [56]:
df_6184.head()

,Date,Teller Id,Transaction Code,Description,Cheque No.,Amount,Tax,Balance,Init Br.
0,04/02/2026 18:32:08,ITBANK,PBDDT,TR fr 2480421139 KIATCHAI AUTO PART 2007,NaN,190000.0,0,703701.57,248
1,04/02/2026 18:34:57,ITBANK,IORDWT,TR to 3253014733 WEISUNTHAI 202602040050,NaN,-10709.0,0,692992.57,248
2,05/02/2026 18:22:54,90180,CBCA,SBK:11 SBR:503 ICAS INCL R1,10127713.0,-26036.0,0,666956.57,700
3,06/02/2026 18:15:43,90180,CBCA,SBK:11 SBR:503 ICAS INCL R1,10127716.0,-18050.0,0,648906.57,700
4,09/02/2026 19:11:24,90180,CBCA,SBK:11 SBR:503 ICAS INCL R1,10127717.0,-5960.0,0,642946.57,700


In [57]:
df = df_7236

df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

In [58]:
df

/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,วันที่,เวลา/ วันที่มีผล,รายการ,ถอนเงิน,ฝากเงิน,ยอดคงเหลือ,ช่องทาง,รายละเอียด
0,01-02-26,NaN,ยอดยกมา,NaN,NaN,"4,191.89",NaN,NaN
1,01-02-26,12:05,รับโอนเงิน,NaN,"25,120.00","29,311.89",Internet/Mobile SCB,จาก SCB X3875 นางสาว ธัญญพัทธ์ ท++
2,02-02-26,09:35,รับโอนเงิน,NaN,"62,506.50","91,818.39",Internet/Mobile KTB,จาก KTB X2446 NARUMON WITHAYAPAL++
3,02-02-26,09:37,โอนเงิน,"90,000.00",NaN,"1,818.39",K BIZ,โอนไป X3557 บจก. เกียรติชัยอะไ++
4,02-02-26,17:58,รับโอนเงิน,NaN,"141,552.50","143,370.89",Internet/Mobile KTB,จาก KTB X2446 NARUMON WITHAYAPAL++
...,...,...,...,...,...,...,...,...
87,28-02-26,11:52,รับโอนเงิน,NaN,"54,444.01","67,230.29",Internet/Mobile KTB,จาก KTB X3316 AUGKANA WILAINIKH++
88,28-02-26,15:34,รับโอนเงิน,NaN,"67,340.50","134,570.79",Internet/Mobile KTB,จาก KTB X2446 NARUMON WITHAYAPAL++
89,28-02-26,15:37,รับโอนเงิน,NaN,"69,815.00","204,385.79",Internet/Mobile KTB,จาก KTB X2446 NARUMON WITHAYAPAL++
90,28-02-26,15:49,โอนเงิน,"200,000.00",NaN,"4,385.79",K BIZ,โอนไป X3557 บจก. เกียรติชัยอะไ++
